In [1]:
import pandas as pd
import numpy as np

In [2]:
movies=pd.read_csv(r"C:\Users\siddh\OneDrive\Desktop\movie-reco\data\movies.csv")
ratings=pd.read_csv(r"C:\Users\siddh\OneDrive\Desktop\movie-reco\data\ratings.csv")
tags=pd.read_csv(r"C:\Users\siddh\OneDrive\Desktop\movie-reco\data\tags.csv")
links=pd.read_csv(r"C:\Users\siddh\OneDrive\Desktop\movie-reco\data\links.csv")

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
movies['genres'] = movies['genres'].fillna('')
movies = movies.merge(links, on='movieId')
tag_data = tags.groupby('movieId')['tag'].apply(
    lambda x: " ".join(x.astype(str))
)
movies = movies.merge(tag_data, on='movieId', how='left')
movies['genres'] = movies['genres'].fillna('')
movies['tag'] = movies['tag'].fillna('')
movies['content'] = (
    movies['genres'] + " " +
    movies['genres'] + " " +
    movies['genres'] + " " +
    movies['tag']
)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['content'])
tfidf_matrix.shape
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim.shape


(9742, 9742)

In [4]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()
def search_movie(name):
    return movies[movies['title'].str.contains(name, case=False)]
def recommend(movie_title, cosine_sim=cosine_sim):
    
    if movie_title not in indices:
        return "Movie not found"
    
    idx = indices[movie_title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:11]
    
    movie_indices = [i[0] for i in sim_scores]
    
    return movies[['title', 'genres','tag']].iloc[movie_indices]

In [5]:
recommend('Jumanji (1995)')


,title,genres,tag
53,"Indian in the Cupboard, The (1995)",Adventure|Children|Fantasy,
109,"NeverEnding Story III, The (1994)",Adventure|Children|Fantasy,
767,Escape to Witch Mountain (1975),Adventure|Children|Fantasy,
1514,Darby O'Gill and the Little People (1959),Adventure|Children|Fantasy,
1556,Return to Oz (1985),Adventure|Children|Fantasy,
1617,"NeverEnding Story, The (1984)",Adventure|Children|Fantasy,
1618,"NeverEnding Story II: The Next Chapter, The (1...",Adventure|Children|Fantasy,
1799,Santa Claus: The Movie (1985),Adventure|Children|Fantasy,
6389,Bridge to Terabithia (2007),Adventure|Children|Fantasy,
6629,"Golden Compass, The (2007)",Adventure|Children|Fantasy,


In [8]:
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)
user_movie_matrix = user_movie_matrix.fillna(0)

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix)
user_similarity.shape
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)
user_similarity_df.iloc[0].sort_values(ascending=False).head(10)

userId
1      1.000000
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
469    0.330664
39     0.329782
288    0.329700
452    0.328048
Name: 1, dtype: float64

In [15]:
target_user = 1
similar_users = user_similarity_df[target_user].sort_values(
    ascending=False
)
similar_users = similar_users.drop(target_user)
top_users = similar_users.head(10)
watched_movies = user_movie_matrix.loc[target_user]

watched_movies = watched_movies[watched_movies > 0].index
recommendations = {}

for user, similarity_score in top_users.items():

    user_ratings = user_movie_matrix.loc[user]

    for movie_id, rating in user_ratings.items():

        if movie_id not in watched_movies and rating > 4:

            if movie_id not in recommendations:
                recommendations[movie_id] = 0

            recommendations[movie_id] += similarity_score * rating
sorted_recommendations = sorted(
    recommendations.items(),
    key=lambda x: x[1],
    reverse=True
)
top_movie_ids = [movie[0] for movie in sorted_recommendations[:10]]
movies[movies['movieId'].isin(top_movie_ids)][['title', 'genres']]

In [18]:
def get_content_scores(movie_title):

    idx = indices[movie_title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    return sim_scores[1:51]

def collaborative_scores(user_id):

    similar_users = user_similarity_df[user_id].sort_values(
        ascending=False
    )

    similar_users = similar_users.drop(user_id)

    top_users = similar_users.head(10)

    watched_movies = user_movie_matrix.loc[user_id]
    watched_movies = watched_movies[watched_movies > 0].index

    recommendations = {}

    for user, similarity_score in top_users.items():

        user_ratings = user_movie_matrix.loc[user]

        for movie_id, rating in user_ratings.items():

            if movie_id not in watched_movies and rating > 4:

                if movie_id not in recommendations:
                    recommendations[movie_id] = 0

                recommendations[movie_id] += (
                    similarity_score * rating
                )

    return recommendations
def content_dict(movie_title):

    scores = get_content_scores(movie_title)

    content_scores = {}

    for movie_idx, score in scores:

        movie_id = movies.iloc[movie_idx].movieId

        content_scores[movie_id] = score

    return content_scores
def hybrid_recommendation(user_id, movie_title):

    content_scores = content_dict(movie_title)

    collab_scores = collaborative_scores(user_id)

    hybrid_scores = {}

    for movie_id in content_scores:

        content_score = content_scores.get(movie_id, 0)

        collab_score = collab_scores.get(movie_id, 0)

        final_score = (
            0.4 * content_score
            +
            0.6 * collab_score
        )

        hybrid_scores[movie_id] = final_score

    sorted_movies = sorted(
        hybrid_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    recommended_ids = [
        movie[0]
        for movie in sorted_movies[:10]
    ]

    return movies[
        movies['movieId'].isin(recommended_ids)
    ][['title', 'genres']]

In [19]:


hybrid_recommendation(
    user_id=1,
    movie_title='Toy Story (1995)'
)

,title,genres
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
1757,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
6194,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
7760,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy
